# INV-M365-AX Operations Department Pack v1

## Purpose
Prove that the Operations department may be treated as one bounded workforce pack only when its personas, workflows, approvals, and readiness state are projected from one department-pack authority.

## Lemma Mapping
- L48

## Invariant Support
- invariants/lemmas/L48_m365_operations_department_pack_v1.yaml

## Expected Result
- The Operations pack authority, runtime extraction, and verifier all exist and stay aligned.

## Runtime Source Cells

The functions below are the notebook source of truth for the Operations department-pack runtime. They intentionally read the authoritative registry and derive pack state from persona registry state plus persona accountability.

In [ ]:
# EXPORT
from __future__ import annotations

import os
import re
from pathlib import Path
from typing import Any

import yaml

from smarthaus_common.json_store import JsonStore
from smarthaus_common.persona_accountability import build_persona_accountability
from smarthaus_common.persona_memory import build_persona_work_history


In [ ]:
# EXPORT
def _slugify_department(value: object | None) -> str:
    return re.sub(r"[^a-z0-9]+", "_", str(value or "").strip().lower()).strip("_")


def _default_department_pack_file(department: str) -> Path:
    registry_file = Path(os.getenv("REGISTRY_FILE", "./registry/agents.yaml")).resolve()
    slug = _slugify_department(department)
    return registry_file.with_name(f"department_pack_{slug}_v1.yaml")


def _default_persona_registry_file() -> Path:
    registry_file = Path(os.getenv("REGISTRY_FILE", "./registry/agents.yaml")).resolve()
    return registry_file.with_name("persona_registry_v2.yaml")


def _load_yaml_object(source: Path, label: str) -> dict[str, Any]:
    if not source.exists():
        raise ValueError(f"{label}_missing:{source}")
    with source.open(encoding="utf-8") as handle:
        payload = yaml.safe_load(handle)
    if not isinstance(payload, dict):
        raise ValueError(f"{label}_invalid")
    return payload


In [ ]:
# EXPORT
def validate_department_pack_authority(payload: dict[str, Any]) -> None:
    required_root = {
        "version",
        "last_updated",
        "authority",
        "department",
        "workflow_families",
        "workload_families",
        "pack_states",
        "approval_model",
        "kpis",
        "personas",
        "bounded_claims",
    }
    missing = sorted(required_root - set(payload))
    if missing:
        raise ValueError(f"department_pack_authority_missing_keys:{','.join(missing)}")

    department = payload.get("department")
    if not isinstance(department, dict):
        raise ValueError("department_pack_authority_invalid_department")
    department_id = str(department.get("id") or "").strip()
    if not department_id:
        raise ValueError("department_pack_authority_department_id_missing")

    personas = payload.get("personas")
    if not isinstance(personas, dict) or not personas:
        raise ValueError("department_pack_authority_invalid_personas")

    total_actions = 0
    for persona_id, persona in personas.items():
        if not isinstance(persona, dict):
            raise ValueError(f"department_pack_persona_invalid:{persona_id}")
        for key in {
            "display_name",
            "title",
            "risk_tier",
            "approval_profile",
            "coverage_status",
            "capability_families",
            "workload_families",
            "workflow_families",
            "supported_actions",
        }:
            if key not in persona:
                raise ValueError(f"department_pack_persona_missing:{persona_id}:{key}")
        coverage_status = str(persona.get("coverage_status") or "")
        if coverage_status not in {"registry-backed", "persona-contract-only"}:
            raise ValueError(f"department_pack_persona_invalid_coverage_status:{persona_id}")
        actions = [str(action) for action in (persona.get("supported_actions") or []) if action]
        if coverage_status == "registry-backed" and not actions:
            raise ValueError(f"department_pack_persona_registry_backed_missing_actions:{persona_id}")
        if coverage_status == "persona-contract-only" and actions:
            raise ValueError(f"department_pack_persona_contract_only_has_actions:{persona_id}")
        total_actions += len(actions)

    kpis = payload.get("kpis")
    if not isinstance(kpis, dict):
        raise ValueError("department_pack_authority_invalid_kpis")
    if int(kpis.get("supported_action_count", -1)) != total_actions:
        raise ValueError("department_pack_authority_supported_action_count_mismatch")
    if int(kpis.get("required_personas", -1)) != len(personas):
        raise ValueError("department_pack_authority_required_persona_count_mismatch")

    pack_states = payload.get("pack_states")
    if not isinstance(pack_states, dict) or set(pack_states) != {
        "ready",
        "watch",
        "attention_required",
        "blocked",
    }:
        raise ValueError("department_pack_authority_invalid_pack_states")


In [ ]:
# EXPORT
def load_department_pack_authority(
    department: str, path: Path | None = None
) -> dict[str, Any]:
    source = path or Path(
        os.getenv("DEPARTMENT_PACK_FILE") or _default_department_pack_file(department)
    )
    payload = _load_yaml_object(source, "department_pack_authority")
    validate_department_pack_authority(payload)
    expected = _slugify_department(payload.get("department", {}).get("id"))
    requested = _slugify_department(department)
    if expected != requested:
        raise ValueError(f"department_pack_department_mismatch:{requested}:{expected}")
    return payload


def _load_persona_registry_payload(path: Path | None = None) -> dict[str, Any]:
    source = path or Path(os.getenv("PERSONA_REGISTRY_FILE") or _default_persona_registry_file())
    payload = _load_yaml_object(source, "persona_registry")
    personas = payload.get("personas")
    if not isinstance(personas, dict) or not personas:
        raise ValueError("persona_registry_missing_personas")
    return payload


def _pack_state_for_personas(personas: list[dict[str, Any]]) -> str:
    if any(str(persona.get("status") or "") != "active" for persona in personas):
        return "blocked"
    accountability_states = [str(persona.get("accountability_state") or "") for persona in personas]
    if any(state == "escalated" for state in accountability_states):
        return "attention_required"
    if any(state == "warning" for state in accountability_states):
        return "watch"
    return "ready"


In [ ]:
# EXPORT
def build_department_pack(
    department: str,
    store: JsonStore | None = None,
    path: Path | None = None,
) -> dict[str, Any]:
    authority = load_department_pack_authority(department, path=path)
    registry = _load_persona_registry_payload()
    runtime_store = store or JsonStore()

    declared_personas = authority.get("personas") or {}
    registry_personas = registry.get("personas") or {}
    projected_personas: list[dict[str, Any]] = []
    workload_families: list[str] = []
    capability_families: list[str] = []

    for persona_id, declared in declared_personas.items():
        registry_persona = registry_personas.get(persona_id)
        if not isinstance(registry_persona, dict):
            raise ValueError(f"department_pack_persona_missing_from_registry:{persona_id}")
        if _slugify_department(registry_persona.get("department")) != _slugify_department(department):
            raise ValueError(f"department_pack_persona_department_mismatch:{persona_id}")
        declared_coverage_status = str(declared.get("coverage_status") or "")
        registry_coverage_status = str(registry_persona.get("coverage_status") or "")
        if declared_coverage_status != registry_coverage_status:
            raise ValueError(f"department_pack_persona_coverage_status_mismatch:{persona_id}")
        declared_actions = [str(action) for action in (declared.get("supported_actions") or []) if action]
        registry_actions = [str(action) for action in (registry_persona.get("allowed_actions") or []) if action]
        if declared_actions != registry_actions:
            raise ValueError(f"department_pack_persona_action_mismatch:{persona_id}")

        accountability = build_persona_accountability(persona_id, store=runtime_store)
        history = build_persona_work_history(persona_id, store=runtime_store)

        for family in declared.get("workload_families") or []:
            family_text = str(family)
            if family_text not in workload_families:
                workload_families.append(family_text)
        for family in declared.get("capability_families") or []:
            family_text = str(family)
            if family_text not in capability_families:
                capability_families.append(family_text)

        projected_personas.append(
            {
                "persona_id": persona_id,
                "display_name": registry_persona.get("display_name"),
                "title": registry_persona.get("title"),
                "status": registry_persona.get("status"),
                "coverage_status": registry_persona.get("coverage_status"),
                "approval_profile": registry_persona.get("approval_profile"),
                "risk_tier": registry_persona.get("risk_tier"),
                "allowed_domains": list(registry_persona.get("allowed_domains") or []),
                "action_count": int(registry_persona.get("action_count") or 0),
                "supported_actions": declared_actions,
                "workflow_families": list(declared.get("workflow_families") or []),
                "capability_families": list(declared.get("capability_families") or []),
                "accountability_state": accountability.get("accountability_state"),
                "queue_depth": accountability.get("metrics", {}).get("queue_depth"),
                "memory_count": history.get("memory_count"),
                "history_events": history.get("event_count"),
            }
        )

    pack_state = _pack_state_for_personas(projected_personas)
    supported_action_count = sum(int(persona.get("action_count") or 0) for persona in projected_personas)
    active_persona_count = sum(1 for persona in projected_personas if persona.get("status") == "active")
    registry_backed_persona_count = sum(
        1 for persona in projected_personas if persona.get("coverage_status") == "registry-backed"
    )
    if supported_action_count != int(authority.get("kpis", {}).get("supported_action_count", -1)):
        raise ValueError("department_pack_supported_action_count_runtime_mismatch")

    return {
        "department": dict(authority.get("department") or {}),
        "approval_model": dict(authority.get("approval_model") or {}),
        "kpis": dict(authority.get("kpis") or {}),
        "workflow_families": list(authority.get("workflow_families") or []),
        "workload_families": list(authority.get("workload_families") or []),
        "capability_families": capability_families,
        "personas": projected_personas,
        "summary": {
            "persona_count": len(projected_personas),
            "active_persona_count": active_persona_count,
            "registry_backed_persona_count": registry_backed_persona_count,
            "supported_action_count": supported_action_count,
            "workload_family_count": len(authority.get("workload_families") or []),
            "workflow_family_count": len(authority.get("workflow_families") or []),
            "pack_state": pack_state,
        },
        "bounded_claims": dict(authority.get("bounded_claims") or {}),
    }


## Verification Cells

These assertions prove the declared Operations boundary resolves cleanly and produces the fixed baseline summary for a clean runtime state.

In [ ]:
root = Path.cwd()
authority = load_department_pack_authority(
    "operations",
    path=root / "registry" / "department_pack_operations_v1.yaml",
)
pack = build_department_pack("operations")

assert authority["department"]["id"] == "operations"
assert pack["summary"]["persona_count"] == 2
assert pack["summary"]["supported_action_count"] == 43
assert pack["summary"]["pack_state"] in {"ready", "watch", "attention_required", "blocked"}
